# ForecastEx

## Web REST API

In [2]:
import requests, json, os
from pprint import pprint

## Get the current markets

In [3]:
from get_forecastex_markets import get_forecastex_markets

In [4]:
fn_market = 'forecastex_markets.json'
if os.path.exists(fn_market):
    with open(fn_market, 'r') as f:
        markets = json.load(f)
else:
    response = requests.get(
        'https://localhost:5000/v1/api/trsrv/event/category-tree',
        verify=False  # skip SSL verification for local gateway
    )
    markets=get_forecastex_markets(response.json())
    with open('forecastex_markets.json', 'w') as f:
        json.dump(markets, f, indent=4)

## First tries get market data

In [22]:
# https://chatgpt.com/share/687ef455-5f4c-8006-946c-4de723e029f1

In [5]:
market = [x for x in markets if x['symbol'] == 'MNYCG'][0]
market

{'symbol': 'MNYCG',
 'conid': 796056051,
 'name': 'General Election for New York City Mayor'}

In [6]:
def check_forecastex_setup():
    base_url = "https://localhost:5000/v1/api"
    
    # Check authentication
    auth_resp = requests.get(f"{base_url}/iserver/auth/status", verify=False)
    auth_data = auth_resp.json()
    print(f"Authentication: {auth_data.get('authenticated', False)}")
    
    # Check account features
    account_resp = requests.get(f"{base_url}/iserver/accounts", verify=False)
    account_data = account_resp.json()
    
    if 'allowFeatures' in account_data:
        features = account_data['allowFeatures']
        event_trading = features.get('allowEventTrading', False)
        event_contracts = features.get('allowEventContract', False)
        
        print(f"Event Trading Enabled: {event_trading}")
        print(f"Event Contracts Enabled: {event_contracts}")
        
        if not event_trading or not event_contracts:
            print("⚠️ ForecastEx permissions NOT enabled!")
            print("→ Go to Client Portal > Settings > Trading Permissions")

if __name__ == "__main__":
    check_forecastex_setup()

ConnectionError: HTTPSConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /v1/api/iserver/auth/status (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x71bdaa6166f0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [39]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import requests
import urllib3
import time
from datetime import datetime

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://localhost:5000/v1/api"
PARENT_CONID = 796056051  # MNYCG parent container
MARKET_NAME = "NYC Mayor Election"
FIELD_CODES = '31,84,85,86,88,7059,6509,72'
MAX_OFFSET = 25  # We'll scan conid+1 to conid+25

session = requests.Session()
session.verify = False  # For localhost
session.timeout = 10

# ──────────────────────────────────────────────────────────────────────────────
# STEP 1: Check authentication and permissions
# ──────────────────────────────────────────────────────────────────────────────
def check_auth():
    resp = session.get(f"{BASE_URL}/iserver/auth/status")
    data = resp.json()
    print("\n🔐 Auth Check:", data)
    if not data.get("authenticated") or not data.get("connected"):
        raise Exception("❌ Client is not authenticated or connected. Please login on https://localhost:5000")
    return True

def check_permissions():
    print("🔎 Checking ForecastEx permissions...")
    account_resp = session.get(f"{BASE_URL}/iserver/accounts")
    if account_resp.status_code != 200:
        print("⚠️ Could not check account features.")
        return
    acc_data = account_resp.json()
    features = acc_data[0] if isinstance(acc_data, list) else acc_data
    # Permissions indicators can vary — placeholder:
    print(f"✅ Account permissions retrieved. (Check Client Portal for Event Contract trading if needed)")

# ──────────────────────────────────────────────────────────────────────────────
# STEP 2: Discover individual candidate contracts via conid offset scan
# ──────────────────────────────────────────────────────────────────────────────
def discover_candidate_conids(parent_conid):
    print("\n🔍 Discovering individual candidate contracts...")
    discovered = []
    for offset in range(1, MAX_OFFSET + 1):
        conid = parent_conid + offset
        data = get_market_data(conid, minimal=True)
        if data and data.get('conid') == conid:
            if any(k in data for k in ['31', '84', '85', '86', '88']):
                print(f"  ✅ Found candidate contract: ConID {conid}")
                discovered.append(conid)
        time.sleep(0.1)  # Avoid rate limit
    return discovered

# ──────────────────────────────────────────────────────────────────────────────
# STEP 3: Market Data Subscription & Retrieval
# ──────────────────────────────────────────────────────────────────────────────
def get_market_data(conid, minimal=False):
    fields = '31,6509' if minimal else FIELD_CODES
    try:
        resp = session.get(
            f"{BASE_URL}/iserver/marketdata/snapshot",
            params={'conids': str(conid), 'fields': fields}
        )
        if resp.status_code == 200:
            return resp.json()[0] if isinstance(resp.json(), list) else resp.json()
        else:
            print(f"    ⚠️ Market data error for {conid}: {resp.status_code}")
    except Exception as e:
        print(f"    ❌ Error retrieving market data for {conid}: {e}")
    return {}

# ──────────────────────────────────────────────────────────────────────────────
# STEP 4: Pretty Output Formatter
# ──────────────────────────────────────────────────────────────────────────────
def format_price(value):
    try:
        return f"${float(value):.3f}"
    except:
        return "N/A"

def display_market_data(conid_list):
    print("\n📈 LIVE MARKET DATA REPORT")
    print("────────────────────────────────────────────────────────────────────────────")
    print(f"{'ConID':<12} {'Last':<8} {'Prob%':<8} {'Bid/Ask':<15} {'Volume':<10} {'Status':<7}")
    print("────────────────────────────────────────────────────────────────────────────")

    for conid in conid_list:
        data = get_market_data(conid)
        last = format_price(data.get("31"))
        bid = format_price(data.get("84"))
        ask = format_price(data.get("86"))

        try:
            prob = f"{float(data.get('31', 0)) * 100:.1f}%"
        except:
            prob = "N/A"

        volume = data.get("72", "—")
        status = data.get("6509", "—")
        ba_spread = f"{bid}/{ask}" if bid != 'N/A' and ask != 'N/A' else 'N/A'

        print(f"{str(conid):<12} {last:<8} {prob:<8} {ba_spread:<15} {volume:<10} {status:<7}")
        time.sleep(0.1)

# ──────────────────────────────────────────────────────────────────────────────
# Main Execution
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"\n🗳️  FORECASTEX {MARKET_NAME.upper()} MARKET SCAN")
    print(f"{'='*72}")
    print(f"Parent ConID: {PARENT_CONID}")
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"Timestamp: {current_time} EST")

    try:
        check_auth()
        check_permissions()
        conids = discover_candidate_conids(PARENT_CONID)

        if not conids:
            print("❌ No individual candidate contracts returned live pricing.")
            print("→ You may be outside market hours or missing ForecastEx permissions.")
        else:
            print(f"\n✅ Discovered {len(conids)} possible ForecastEx candidate contracts.")
            display_market_data(conids)

    except Exception as err:
        print(f"‼️ Error during analysis: {err}")



🗳️  FORECASTEX NYC MAYOR ELECTION MARKET SCAN
Parent ConID: 796056051
Timestamp: 2025-07-21 22:26:02 EST

🔐 Auth Check: {'authenticated': True, 'competing': False, 'connected': True, 'message': '', 'MAC': '98:F2:B3:23:AE:D0', 'serverInfo': {'serverName': 'JifN17073', 'serverVersion': 'Build 10.38.1a, Jul 15, 2025 11:26:07 AM'}, 'fail': ''}
🔎 Checking ForecastEx permissions...
✅ Account permissions retrieved. (Check Client Portal for Event Contract trading if needed)

🔍 Discovering individual candidate contracts...
❌ No individual candidate contracts returned live pricing.
→ You may be outside market hours or missing ForecastEx permissions.


In [44]:
cookies = {
    "PHPSESSID": "a3d1pgje737171nrjo808f50j3",  # from your list
    "ibcust": "75058fe1e784851d4944d13ef25a65fc",
    "IS_MASTER": "false",
    "ft.lb": "n1.990635d8a0f0a8bde1f833240434596af7514b26",
    "ROUTEIDD": ".ny5japp1",
    "RT": "z=1&dm=forecasttrader.interactivebrokers.com&si=...&ss=...&sl=...&tt=..."
}

import requests

# API endpoint for binary option (yes/no) market data
url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"

# Candidate contracts (you can paste up to 20+ here)
conids = ["796056520", "796056525"]
params = {
    "conids": ",".join(conids)
}

# Send the GET request
response = requests.get(url, params=params, cookies=cookies)

# Output result
if response.status_code == 200:
    for item in response.json():
        print(f"{item['longDescription']}: ${item['last']} → Bid: {item['bid']} / Ask: {item['ask']} Vol: {item['volume']}")
else:
    print(f"⚠️ HTTP {response.status_code}: {response.text}")


KeyError: 'last'

In [43]:
pprint(response.json())

[{'categories': ['g17549', 'g17469', 'g7428'],
  'commodityCode': 'MNYCG',
  'conid': 796056520,
  'currency': 'USD',
  'eventAuthorityURL': 'https://www.vote.nyc/',
  'eventFixedPayout': '1',
  'exchange': 'FORECASTX',
  'expectedPayoutTime': '20251130130000',
  'expectedResolutionTime': '20251129165959',
  'expiration': '20251129',
  'lastTradeDate': '20251129',
  'lastTradeMillis': 1764457140000,
  'lastTradeTime': '1659',
  'longDescription': 'Will Zohran Mamdani win the New York City general '
                     'election for mayor in 2025?',
  'market': 'New York City',
  'marketRulesLink': 'https://data.forecastex.com/regulatory/MTermsandConditions.pdf',
  'name': "MNYCG Nov04'25 Mamdani",
  'popularityRank': -1,
  'priceIncrement': 0.01,
  'putOrCall': 'C',
  'shortDescription': "MNYCG Nov04'25 Mamdani YES @FORECASTX",
  'sourceAgency': 'Board of Elections in the City of New York',
  'strike': 4.0,
  'strikeLabel': 'Mamdani',
  'timespecifierParam': '2025.11.4',
  'timezone':

In [48]:
import requests

# If you needed full cookie-based access you’d include them like so:
cookies = {
    # optional, most cases for ForecastTrader do not require auth cookies
}

conids = ["796056520", "796056525"]  # Mamdani YES and NO

url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
params = { "conids": ",".join(conids) }

resp = requests.get(url, params=params, cookies=cookies)
data = resp.json()
pprint(data)

[{'categories': ['g17549', 'g17469', 'g7428'],
  'commodityCode': 'MNYCG',
  'conid': 796056520,
  'currency': 'USD',
  'eventAuthorityURL': 'https://www.vote.nyc/',
  'eventFixedPayout': '1',
  'exchange': 'FORECASTX',
  'expectedPayoutTime': '20251130130000',
  'expectedResolutionTime': '20251129165959',
  'expiration': '20251129',
  'lastTradeDate': '20251129',
  'lastTradeMillis': 1764457140000,
  'lastTradeTime': '1659',
  'longDescription': 'Will Zohran Mamdani win the New York City general '
                     'election for mayor in 2025?',
  'market': 'New York City',
  'marketRulesLink': 'https://data.forecastex.com/regulatory/MTermsandConditions.pdf',
  'name': "MNYCG Nov04'25 Mamdani",
  'popularityRank': -1,
  'priceIncrement': 0.01,
  'putOrCall': 'C',
  'shortDescription': "MNYCG Nov04'25 Mamdani YES @FORECASTX",
  'sourceAgency': 'Board of Elections in the City of New York',
  'strike': 4.0,
  'strikeLabel': 'Mamdani',
  'timespecifierParam': '2025.11.4',
  'timezone':

In [49]:
import requests

# Candidate contracts ConIDs
conids = [
    796056520,  # Mamdani YES
    796056525   # Mamdani NO
]

endpoint = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
params = {
    "conids": ",".join(map(str, conids))
}

# You can often call this without authentication cookies (browser guest works)
r = requests.get(endpoint, params=params)

if r.status_code == 200:
    for c in r.json():
        print(f"{c.get('contractDisplayName', '—')}:")
        print(f"  ✅ Last: {c.get('last')}")
        print(f"  ✅ Bid:  {c.get('bid')}")
        print(f"  ✅ Ask:  {c.get('ask')}")
        print(f"  🔁 Vol:  {c.get('volume')}")
else:
    print(f"❌ Error: HTTP {r.status_code}")
    print(r.text)


—:
  ✅ Last: None
  ✅ Bid:  None
  ✅ Ask:  None
  🔁 Vol:  None
—:
  ✅ Last: None
  ✅ Bid:  None
  ✅ Ask:  None
  🔁 Vol:  None


In [51]:
import requests

market_id = "796056051|20251129|4"
info_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/GetContractInfo"
r = requests.get(info_url, params={"marketId": market_id})
contracts = r.json()

yes_conid = None
no_conid = None

for c in contracts:
    if c.get("strikeLabel") == "Sliwa":
        if c.get("putOrCall") == "C":
            yes_conid = c["conid"]
        elif c.get("putOrCall") == "P":
            no_conid = c["conid"]

print("Sliwa YES:", yes_conid)
print("Sliwa NO: ", no_conid)


AttributeError: 'str' object has no attribute 'get'

In [52]:
contracts

{'error': 'Not found'}

In [ ]:
import requests
from collections import defaultdict

# -- Main market ID for NYC Mayor Election
market_conid = "796056051"
market_url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/contracts"
pricing_url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"

# -- Step 1: Get all contracts under the market
def fetch_all_contracts():
    params = {
        "showrestricted": "false",
        "market": market_conid
    }
    r = requests.get(market_url, params=params)
    r.raise_for_status()
    contracts = r.json()["contracts"]
    return contracts

# -- Step 2: Group contracts by candidate with YES and NO
def extract_candidate_conids(contract_data):
    candidates = defaultdict(dict)
    for c in contract_data:
        name = c.get("strikeLabel")
        direction = "YES" if c.get("putOrCall") == "C" else "NO"
        candidates[name][direction] = {
            "conid": c["conid"],
            "description": c["shortDescription"]
        }
    return candidates

# -- Step 3: Fetch pricing data from binaryoptions endpoint
def fetch_market_data(conids):
    params = {
        "conids": ",".join(str(c) for c in conids)
    }
    r = requests.get(pricing_url, params=params)
    r.raise_for_status()
    return r.json()

# -- Step 4: Merge contract metadata with pricing
def merge_prices(candidates, price_data):
    conid_map = {int(p["conid"]): p for p in price_data}

    display = []
    for name, sides in candidates.items():
        row = {"Candidate": name}
        for direction in ['YES', 'NO']:
            if direction in sides:
                cid = sides[direction]["conid"]
                contract = conid_map.get(cid, {})
                row[f"{direction} Price"] = contract.get("last")
                row[f"{direction} Bid"] = contract.get("bid")
                row[f"{direction} Ask"] = contract.get("ask")
                row[f"{direction} Volume"] = contract.get("volume")
        display.append(row)
    return display

# -- Final Step: Display table
def print_table(data):
    print(f"\n🗳️ NYC MAYOR ELECTION MARKET PRICING")
    print("───────────────────────────────────────────────────────────────────────────────")
    for row in data:
        print(f"{row['Candidate']:<12} | YES: ${row.get('YES Price', '—'):<5} "
              f"(Bid: {row.get('YES Bid', '—')} / Ask: {row.get('YES Ask', '—')})"
              f" | NO: ${row.get('NO Price', '—'):<5} "
              f"(Bid: {row.get('NO Bid', '—')} / Ask: {row.get('NO Ask', '—')})")

In [ ]:
    contracts = fetch_all_contracts()
    candidates = extract_candidate_conids(contracts)
    all_conids = [side["conid"] for c in candidates.values() for side in c.values()]
    price_data = fetch_market_data(all_conids)
    table_data = merge_prices(candidates, price_data)

In [63]:
conid_list = [side["conid"] for candidate in candidates.values() for side in candidate.values()]

In [64]:
import requests

url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
params = {"conids": ",".join(map(str, conid_list))}

response = requests.get(url, params=params)
market_data = response.json()

# Build lookup dict by conid
conid_to_price = {int(entry["conid"]): entry for entry in market_data}


In [66]:
results = []

for name, contracts in candidates.items():
    row = {"Candidate": name}

    for side, contract in contracts.items():
        conid = contract["conid"]
        live = conid_to_price.get(conid)

        row[f"{side} Price"] = live.get("last") if live else None
        row[f"{side} Bid"] = live.get("bid") if live else None
        row[f"{side} Ask"] = live.get("ask") if live else None
        row[f"{side} Volume"] = live.get("volume") if live else None

    results.append(row)


In [67]:
results

[{'Candidate': 'Sliwa', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Cuomo', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Walden', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Mamdani', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Adams', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}]

In [68]:
live

{'market': 'New York City', 'popularityRank': -1, 'name': "MNYCG Nov04'25 Adams", 'longDescription': 'Will Eric Adams win the New York City general election for mayor in 2025?', 'putOrCall': 'P', 'expiration': '20251129', 'currency': 'USD', 'lastTradeMillis': 1764457140000, 'lastTradeDate': '20251129', 'lastTradeTime': '1659', 'timezone': 'America/Chicago', 'commodityCode': 'MNYCG', 'eventAuthorityURL': 'https://www.vote.nyc/', 'eventFixedPayout': '1', 'sourceAgency': 'Board of Elections in the City of New York', 'marketRulesLink': 'https://data.forecastex.com/regulatory/MTermsandConditions.pdf', 'underlyingName': 'General Election for New York City Mayor', 'categories': ['g17549', 'g17469', 'g7428'], 'expectedResolutionTime': '20251129165959', 'expectedPayoutTime': '20251130130000', 'timespecifierParam': '2025.11.4', 'exchange': 'FORECASTX', 'priceIncrement': 0.01, 'tradingHours': {'timezone': 'America/Chicago', 'holidays': []}, 'conid': 796056534, 'underlyingConid': 796056051, 'under

In [69]:
row

{'Candidate': 'Adams', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}

In [70]:
import requests
from collections import defaultdict

# ✅ These are discovered from the /contracts endpoint
candidates = {
    "Sliwa": {
        "YES": {"conid": 796056496, "description": "YES"},
        "NO":  {"conid": 796056501, "description": "NO"},
    },
    "Cuomo": {
        "YES": {"conid": 796056506, "description": "YES"},
        "NO":  {"conid": 796056511, "description": "NO"},
    },
    "Walden": {
        "YES": {"conid": 796056514, "description": "YES"},
        "NO":  {"conid": 796056519, "description": "NO"},
    },
    "Mamdani": {
        "YES": {"conid": 796056520, "description": "YES"},
        "NO":  {"conid": 796056525, "description": "NO"},
    },
    "Adams": {
        "YES": {"conid": 796056531, "description": "YES"},
        "NO":  {"conid": 796056534, "description": "NO"},
    },
}

# 📌 Build list of all unique ConIDs
conids = [val["conid"] for c in candidates.values() for val in c.values()]

# 🎯 Step 1: Get live price data from /binaryoptions
def get_price_data(conids):
    url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
    params = {"conids": ",".join(map(str, conids))}
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

# 🔄 Step 2: Merge candidates with associated pricing
def merge_prices(candidates, price_data):
    conid_lookup = {int(entry["conid"]): entry for entry in price_data}
    results = []

    for name, sides in candidates.items():
        row = {"Candidate": name}
        for side in ["YES", "NO"]:
            conid = sides[side]["conid"]
            live = conid_lookup.get(conid, {})
            row[f"{side} Price"] = live.get("last")
            row[f"{side} Bid"] = live.get("bid")
            row[f"{side} Ask"] = live.get("ask")
            row[f"{side} Volume"] = live.get("volume")
        results.append(row)

    return results

In [73]:
conids

[796056496, 796056501, 796056506, 796056511, 796056514, 796056519, 796056520, 796056525, 796056531, 796056534]

In [ ]:
# ✅ Run everything

price_data = get_price_data(conids)

In [74]:
price_data

[{'market': 'New York City', 'popularityRank': -1, 'name': "MNYCG Nov04'25 Sliwa", 'longDescription': 'Will Curtis Sliwa win the New York City general election for mayor in 2025?', 'putOrCall': 'C', 'expiration': '20251129', 'currency': 'USD', 'lastTradeMillis': 1764457140000, 'lastTradeDate': '20251129', 'lastTradeTime': '1659', 'timezone': 'America/Chicago', 'commodityCode': 'MNYCG', 'eventAuthorityURL': 'https://www.vote.nyc/', 'eventFixedPayout': '1', 'sourceAgency': 'Board of Elections in the City of New York', 'marketRulesLink': 'https://data.forecastex.com/regulatory/MTermsandConditions.pdf', 'underlyingName': 'General Election for New York City Mayor', 'categories': ['g17549', 'g17469', 'g7428'], 'expectedResolutionTime': '20251129165959', 'expectedPayoutTime': '20251130130000', 'timespecifierParam': '2025.11.4', 'exchange': 'FORECASTX', 'priceIncrement': 0.01, 'tradingHours': {'timezone': 'America/Chicago', 'holidays': []}, 'conid': 796056496, 'underlyingConid': 796056051, 'un

In [75]:
table = merge_prices(candidates, price_data)

In [76]:
table

[{'Candidate': 'Sliwa', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Cuomo', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Walden', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Mamdani', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}, {'Candidate': 'Adams', 'YES Price': None, 'YES Bid': None, 'YES Ask': None, 'YES Volume': None, 'NO Price': None, 'NO Bid': None, 'NO Ask': None, 'NO Volume': None}]

In [ ]:
print(f"\n🗳️ NYC Mayor 2025 – Live Forecast Market\n")
for row in table:
    print(
        f"{row['Candidate']:10s} | "
        f"YES ${row['YES Price']}, Bid: {row['YES Bid']}, Ask: {row['YES Ask']} | "
        f"NO ${row['NO Price']}, Bid: {row['NO Bid']}, Ask: {row['NO Ask']}"
    )


🗳️ NYC Mayor 2025 – Live Forecast Market

Sliwa      | YES $None, Bid: None, Ask: None | NO $None, Bid: None, Ask: None
Cuomo      | YES $None, Bid: None, Ask: None | NO $None, Bid: None, Ask: None
Walden     | YES $None, Bid: None, Ask: None | NO $None, Bid: None, Ask: None
Mamdani    | YES $None, Bid: None, Ask: None | NO $None, Bid: None, Ask: None
Adams      | YES $None, Bid: None, Ask: None | NO $None, Bid: None, Ask: None


In [77]:
import requests

candidates = {
    "Sliwa": {"YES": 796056496, "NO": 796056501},
    "Cuomo": {"YES": 796056506, "NO": 796056511},
    "Walden": {"YES": 796056514, "NO": 796056519},
    "Mamdani": {"YES": 796056520, "NO": 796056525},
    "Adams": {"YES": 796056531, "NO": 796056534},
}

conids = [conid for pair in candidates.values() for conid in pair.values()]
url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"

r = requests.get(url, params={"conids": ",".join(map(str, conids))})
r.raise_for_status()

price_data = r.json()
print("✅ Fields in binaryoptions response:", list(price_data[0].keys()))

# Build quick lookup
lookup = {item['conid']: item for item in price_data}

# Merge for display
for name, pair in candidates.items():
    yes = lookup.get(pair["YES"], {})
    no = lookup.get(pair["NO"], {})
    print(f"{name:10} | YES: ${yes.get('last')} | NO: ${no.get('last')}")


✅ Fields in binaryoptions response: ['market', 'popularityRank', 'name', 'longDescription', 'putOrCall', 'expiration', 'currency', 'lastTradeMillis', 'lastTradeDate', 'lastTradeTime', 'timezone', 'commodityCode', 'eventAuthorityURL', 'eventFixedPayout', 'sourceAgency', 'marketRulesLink', 'underlyingName', 'categories', 'expectedResolutionTime', 'expectedPayoutTime', 'timespecifierParam', 'exchange', 'priceIncrement', 'tradingHours', 'conid', 'underlyingConid', 'underlyingSymbol', 'shortDescription', 'strike', 'strikeLabel']
Sliwa      | YES: $None | NO: $None
Cuomo      | YES: $None | NO: $None
Walden     | YES: $None | NO: $None
Mamdani    | YES: $None | NO: $None
Adams      | YES: $None | NO: $None


In [78]:
import requests
from datetime import datetime

def get_candidate_probability(conid, period="1week"):
    url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"
    params = {
        "conid": conid,
        "period": period,
        "exchange": "FORECASTX",
        "secType": "OPT"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    timestamps = data.get("time", [])
    prices = data.get("avg", [])

    if not prices or not timestamps:
        return None

    latest_price = float(prices[-1])
    latest_time = datetime.utcfromtimestamp(int(timestamps[-1]))

    return {
        "conid": conid,
        "probability_pct": latest_price * 100,
        "timestamp": latest_time.isoformat() + "Z"
    }


In [79]:
result = get_candidate_probability(796056520)  # Mamdani YES conid
print(f"✅ Mamdani latest probability: {result['probability_pct']:.1f}% (as of {result['timestamp']})")

✅ Mamdani latest probability: 71.0% (as of 2025-07-21T20:30:00Z)


/tmp/ipykernel_39034/2880084448.py:24: DeprecationWarning:

datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).



## Final version get market data

In [80]:
market = {'conid': 796056051, 'name': 'General Election for New York City Mayor', 'symbol': 'MNYCG'}

### Discover All YES/NO Candidate Subcontracts

In [82]:
import requests

# Market container for NYC 2025
market_conid = str(market["conid"])
contracts_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/contracts"
contracts_params = {"showrestricted":"false", "market":market_conid}

contracts_resp = requests.get(contracts_url, params=contracts_params)
contracts_resp.raise_for_status()
contracts_raw = contracts_resp.json()["contracts"]

# Organize by candidate and side
from collections import defaultdict
candidates = defaultdict(dict)
for c in contracts_raw:
    name = c["strikeLabel"]
    side = "YES" if c["putOrCall"] == "C" else "NO"
    candidates[name][side] = c["conid"]

print("CANDIDATES & CONIDs:")
for name, sides in candidates.items():
    print(f"{name:10} YES: {sides['YES']} NO: {sides['NO']}")


CANDIDATES & CONIDs:
Sliwa      YES: 796056496 NO: 796056501
Cuomo      YES: 796056506 NO: 796056511
Walden     YES: 796056514 NO: 796056519
Mamdani    YES: 796056520 NO: 796056525
Adams      YES: 796056531 NO: 796056534


###  Get LIVE PRICING for All Candidates

In [83]:
# Build list of all YES and NO conids
conid_list = []
for sides in candidates.values():
    conid_list.extend([sides["YES"], sides["NO"]])

pricing_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
pricing_params = {"conids": ",".join(str(cid) for cid in conid_list)}

pricing_resp = requests.get(pricing_url, params=pricing_params)
pricing_resp.raise_for_status()
live_data = pricing_resp.json()

# Build a lookup dict by conid
live_lookup = {d["conid"]: d for d in live_data}

### Get LATEST HISTORICAL "LINE CHART" VALUE for Each YES Contract

In [86]:
def get_latest_probability(conid):
    """Returns the latest YES probability from forecastContract chart, or None if unavailable."""
    url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"
    params = {
        "conid": conid,
        "period": "1week",
        "exchange": "FORECASTX",
        "secType": "OPT"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    avg = data.get("avg")
    if not avg:
        return None
    return round(avg[-1] * 100, 2)



### Print the Full Per-Candidate Table (Current, Bid/Ask, Historical Line)

In [87]:
print("\nNYC Mayor Election Market — ForecastEx Live Snapshot")
print(f"{'Candidate':10s}  {'YES':>6s}  {'NO':>6s}  {'YES-bid':>8s}  {'YES-ask':>8s}  {'NO-bid':>7s}  {'NO-ask':>7s}  {'YES Line%':>9s}")

for name, sides in candidates.items():
    yes = live_lookup.get(sides["YES"], {})
    no  = live_lookup.get(sides["NO"], {})
    # Most recent line chart/YES-probability
    yes_prob = get_latest_probability(sides["YES"])

    line_str = f"{yes_prob:>9.2f}" if yes_prob is not None else "    —    "

    print(f"{name:10s}  "
          f"{yes.get('last', '—'):>6}  "
          f"{no.get('last', '—'):>6}  "
          f"{yes.get('bid', '—'):>8}  "
          f"{yes.get('ask', '—'):>8}  "
          f"{no.get('bid', '—'):>7}  "
          f"{no.get('ask', '—'):>7}  "
          f"{line_str}")


NYC Mayor Election Market — ForecastEx Live Snapshot
Candidate      YES      NO   YES-bid   YES-ask   NO-bid   NO-ask  YES Line%
Sliwa            —       —         —         —        —        —       7.00
Cuomo            —       —         —         —        —        —      18.00
Walden           —       —         —         —        —        —      —    
Mamdani          —       —         —         —        —        —      71.00
Adams            —       —         —         —        —        —      11.00


In [88]:
# Example output:
candidates = {
    'Mamdani': {'YES': 796056520, 'NO': 796056525},
    'Adams': {'YES': 796056531, 'NO': 796056534},
    # etc.
}


In [89]:
import requests

conids = [cid for sides in candidates.values() for cid in sides.values()]
url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
resp = requests.get(url, params={"conids": ",".join(map(str, conids))})
resp.raise_for_status()
live_data = resp.json()

In [91]:
pprint(live_data)

[{'categories': ['g17549', 'g17469', 'g7428'],
  'commodityCode': 'MNYCG',
  'conid': 796056520,
  'currency': 'USD',
  'eventAuthorityURL': 'https://www.vote.nyc/',
  'eventFixedPayout': '1',
  'exchange': 'FORECASTX',
  'expectedPayoutTime': '20251130130000',
  'expectedResolutionTime': '20251129165959',
  'expiration': '20251129',
  'lastTradeDate': '20251129',
  'lastTradeMillis': 1764457140000,
  'lastTradeTime': '1659',
  'longDescription': 'Will Zohran Mamdani win the New York City general '
                     'election for mayor in 2025?',
  'market': 'New York City',
  'marketRulesLink': 'https://data.forecastex.com/regulatory/MTermsandConditions.pdf',
  'name': "MNYCG Nov04'25 Mamdani",
  'popularityRank': -1,
  'priceIncrement': 0.01,
  'putOrCall': 'C',
  'shortDescription': "MNYCG Nov04'25 Mamdani YES @FORECASTX",
  'sourceAgency': 'Board of Elections in the City of New York',
  'strike': 4.0,
  'strikeLabel': 'Mamdani',
  'timespecifierParam': '2025.11.4',
  'timezone':

In [92]:
lookup = {item['conid']: item for item in live_data}

print(f"{'Candidate':10s}  {'YES %':>6s}  {'NO %':>6s}  {'OI':>8s}")
for name, sides in candidates.items():
    yes = lookup.get(sides['YES'], {})
    no = lookup.get(sides['NO'], {})
    oi = yes.get('openInterest', '—')  # Or whatever OI field is present
    yes_pct = round(yes.get('last', 0) * 100, 2) if 'last' in yes else '—'
    no_pct  = round(no.get('last', 0) * 100, 2) if 'last' in no else '—'
    print(f"{name:10s}  {yes_pct:>6}  {no_pct:>6}  {oi:>8}")

Candidate    YES %    NO %        OI
Mamdani          —       —         —
Adams            —       —         —


In [93]:
import requests

conid = 796056520  # e.g. Mamdani YES
url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/details"
resp = requests.get(url, params={"conid": conid})
pprint(resp.json())


{'error': 'Not found'}


In [94]:
import requests

# Example: Map of candidate names to YES/NO ConIDs
candidates = {
    'Sliwa':   {'YES': 796056496, 'NO': 796056501},
    'Cuomo':   {'YES': 796056506, 'NO': 796056511},
    'Walden':  {'YES': 796056514, 'NO': 796056519},
    'Mamdani': {'YES': 796056520, 'NO': 796056525},
    'Adams':   {'YES': 796056531, 'NO': 796056534},
}

conids = [cid for side in candidates.values() for cid in side.values()]
url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
params = {"conids": ",".join(map(str, conids))}

resp = requests.get(url, params=params)
resp.raise_for_status()
live_data = resp.json()

# Build a lookup: conid -> data dictionary
lookup = {item['conid']: item for item in live_data}

# Print results with OI if present
print(f"{'Candidate':9s}  {'YES %':>6s}  {'NO %':>6s}  {'OI':>9s}")
for name, side_ids in candidates.items():
    yesd = lookup.get(side_ids['YES'], {})
    nod  = lookup.get(side_ids['NO'], {})
    yes_pct = round((yesd.get('last') or 0) * 100) if 'last' in yesd else '—'
    no_pct  = round((nod.get('last') or 0) * 100) if 'last' in nod else '—'
    oi = yesd.get('openInterest') or nod.get('openInterest') or yesd.get('oi') or nod.get('oi') or '—'
    print(f"{name:9s}  {str(yes_pct):>6}  {str(no_pct):>6}  {oi:>9}")


Candidate   YES %    NO %         OI
Sliwa           —       —          —
Cuomo           —       —          —
Walden          —       —          —
Mamdani         —       —          —
Adams           —       —          —
